In [ ]:
from lakehouse_common import create_spark, courier_history
spark = create_spark("Delivery - Courier history")
base = "s3a://lakehouse/warehouse/"
# Use the same accept date and eligible orders as the fact table.
silver = spark.read.format("delta").load(base + "delivery_silver")
history = courier_history(silver)
assert history.filter("effective_date >= end_date").count() == 0
assert history.groupBy("courier_id", "effective_date").count().filter("count > 1").count() == 0
history.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(base + "dim_courier_silver")
history.orderBy("courier_id", "effective_date").show(20, False)